<a href="https://colab.research.google.com/github/rwcitek/TechEx-LangGraph-workshop/blob/main/notebooks/solutions/05_evaluation_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5 — Evaluation & Reliability

> **Time:** 20 minutes.
>
> **What you'll do:** build a 6-example eval suite for the triage system, implement two rule-based scorers and one LLM-as-judge, run them against the baseline, then catch a planted regression.

The eval suite you build here is reusable beyond the workshop — the same pattern works for any agent system.

> 💡  This notebook can run in two modes:
> - **With LangSmith** — uses `langsmith.evaluation.evaluate()` for hosted experiment tracking
> - **Local mode** — uses a simple inline runner that does the same scoring but prints results locally
>
> Both produce the same scorer outputs. We default to local mode so the lab works without a LangSmith account.

> *Solution notebook. Try the starter first.*

## 1.  Setup

In [1]:
%pip install -q \
    langgraph==0.2.* \
    langchain==0.3.* \
    langchain-openai==0.2.* \
    langsmith==0.1.*


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.63 which is incompatibl

In [2]:
import os
from typing import TypedDict, Annotated, Literal
from operator import add
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
import json
from langsmith import Client
from langsmith.evaluation import evaluate


In [3]:
def _ensure_key(name: str, optional: bool = False) -> None:
    """Load an API key from (in order): existing env, Colab Secrets, or getpass.

    Colab Secrets are the recommended path for this workshop — set them ONCE
    via the 🔑 key icon in Colab's left sidebar and every notebook will pick
    them up automatically.
    """
    if os.environ.get(name):
        print(f"  ✓  {name} already set in environment")
        return
    # 1. Try Colab Secrets
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f"  ✓  {name} loaded from Colab Secrets")
            return
    except Exception:
        pass
    # 2. Fallback — prompt the user
    import getpass
    val = getpass.getpass(f"Paste your {name}{' (optional)' if optional else ''}: ").strip()
    if val:
        os.environ[name] = val
        print(f"  ✓  {name} set")
    elif optional:
        print(f"  •  {name} skipped (optional)")
    else:
        print(f"  ⚠   {name} skipped — you'll hit errors later without it")

_ensure_key("OPENAI_API_KEY")
_ensure_key("LANGSMITH_API_KEY", optional=True)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "triage-workshop-eval"

print("Ready.")


  ✓  OPENAI_API_KEY loaded from Colab Secrets
  ✓  LANGSMITH_API_KEY loaded from Colab Secrets
Ready.


## 2.  The baseline system

In [4]:
KB = {
    "billing": [
        {"id": "B-001", "title": "Billing cycle and prorations",
         "text": "We bill on the same day each month. If you change plans mid-cycle, the next invoice is prorated."},
        {"id": "B-002", "title": "Refund policy",
         "text": "Refunds are available within 14 days. Issued to original payment method, settle in 5 business days."},
        {"id": "B-003", "title": "Failed payments",
         "text": "Failed payments retry once a day for 3 days, then 7-day grace period."},
    ],
    "technical": [
        {"id": "T-001", "title": "Login issues",
         "text": "Clear cookies. For MFA failures check spam and verify phone."},
        {"id": "T-002", "title": "API rate limits",
         "text": "Standard plan: 60 requests per minute. Enterprise: 600. Respect Retry-After header."},
        {"id": "T-003", "title": "Export failures",
         "text": "Retry from the same dialog — export jobs are idempotent."},
    ],
    "account": [
        {"id": "A-001", "title": "Password reset",
         "text": "Use Forgot password. Reset email valid for 30 minutes. Check spam."},
        {"id": "A-002", "title": "Account closure",
         "text": "Settings → Account → Close. 30-day pending-deletion window."},
        {"id": "A-003", "title": "Ownership transfer",
         "text": "Owner invites new owner as Admin, both confirm via email."},
    ],
}


class TriageState(TypedDict):
    ticket: str
    category: Literal["billing", "technical", "account"] | None
    urgency:  Literal["low", "med", "high"] | None
    retrieved: list[dict]
    draft: str
    verdict: Literal["pass", "revise"] | None
    revision_count: int
    revisions: Annotated[list[str], add]


def make_initial_state(ticket: str) -> TriageState:
    return {"ticket": ticket, "category": None, "urgency": None,
            "retrieved": [], "draft": "", "verdict": None,
            "revision_count": 0, "revisions": []}


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def parse_json(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"): text = text[4:]
    return json.loads(text.strip())


CLASSIFY_PROMPT = """\
Categorize the ticket as one of: billing, technical, account.
Rate urgency as: low, med, high.
Return JSON: {{"category": "...", "urgency": "..."}}

TICKET: {ticket}
"""


def classify(state):
    response = llm.invoke(CLASSIFY_PROMPT.format(ticket=state["ticket"]))
    p = parse_json(response.content)
    return {"category": p["category"], "urgency": p["urgency"]}


def retrieve(state):
    return {"retrieved": KB.get(state["category"], [])}


GOOD_DRAFT_PROMPT = """\
You are a customer support agent.

Reply to the ticket using ONLY the policies below. If a policy doesn't
answer the question, say so. Do not invent details.

POLICIES:
{context}

TICKET:
{ticket}

Write a concise reply (3-6 sentences). Cite the policy ID when you use one.
"""


def draft(state):
    context = "\n\n".join(f"[{d['id']}] {d['title']}\n{d['text']}"
                            for d in state["retrieved"])
    prompt = GOOD_DRAFT_PROMPT.format(context=context, ticket=state["ticket"])
    r = llm.invoke(prompt)
    return {"draft": r.content, "revisions": [r.content]}


QA_PROMPT = """\
Decide if the DRAFT is acceptable. Return JSON: {{"verdict": "pass" | "revise"}}
TICKET: {ticket}
DRAFT:  {draft}
"""


def qa(state):
    r = llm.invoke(QA_PROMPT.format(ticket=state["ticket"], draft=state["draft"]))
    p = parse_json(r.content)
    return {"verdict": p["verdict"], "revision_count": state["revision_count"] + 1}


MAX_REVISIONS = 2


def route_qa(state):
    if state["verdict"] == "pass": return "END"
    if state["revision_count"] >= MAX_REVISIONS: return "END"
    return "drafter"


def build_graph(draft_fn=draft):
    g = StateGraph(TriageState)
    g.add_node("classify", classify)
    g.add_node("retrieve", retrieve)
    g.add_node("drafter",  draft_fn)
    g.add_node("qa",       qa)
    g.set_entry_point("classify")
    g.add_edge("classify", "retrieve")
    g.add_edge("retrieve", "drafter")
    g.add_edge("drafter",  "qa")
    g.add_conditional_edges("qa", route_qa,
        {"drafter": "drafter", "END": END})
    return g.compile()


app = build_graph()
print("System ready.")

System ready.


## 3.  The eval dataset

Six examples, hand-curated. Three categories, plus a high-urgency case and an edge case. Each example has the input ticket and the expected outputs — including which KB articles *should* be cited.

In [5]:
EVAL_EXAMPLES = [
    {
        "id": "E-001",
        "inputs": {"ticket": "Hi, my monthly bill is $50 higher than last month. Can you check?"},
        "outputs": {
            "category": "billing",
            "urgency":  "med",
            "must_mention": ["proration"],
            "must_cite":    ["B-001"],
        },
    },
    {
        "id": "E-002",
        "inputs": {"ticket": "Charged me twice for the same month. Refund please."},
        "outputs": {
            "category": "billing",
            "urgency":  "high",
            "must_mention": ["refund", "14 days"],
            "must_cite":    ["B-002"],
        },
    },
    {
        "id": "E-003",
        "inputs": {"ticket": "I keep getting 429 errors from your API on the Standard plan."},
        "outputs": {
            "category": "technical",
            "urgency":  "low",
            "must_mention": ["60", "Retry-After"],
            "must_cite":    ["T-002"],
        },
    },
    {
        "id": "E-004",
        "inputs": {"ticket": "Data export failed twice this morning."},
        "outputs": {
            "category": "technical",
            "urgency":  "med",
            "must_mention": ["retry", "idempotent"],
            "must_cite":    ["T-003"],
        },
    },
    {
        "id": "E-005",
        "inputs": {"ticket": "Forgot my password and reset email never came. Demo in 30 minutes!"},
        "outputs": {
            "category": "account",
            "urgency":  "high",
            "must_mention": ["spam", "30 minutes"],
            "must_cite":    ["A-001"],
        },
    },
    {
        "id": "E-006",
        "inputs": {"ticket": "What's the meaning of life? Also can you upgrade my plan to Enterprise?"},
        "outputs": {
            "category": "billing",
            "urgency":  "low",
            "must_mention": ["upgrade"],
            "must_cite":    [],
        },
    },
]
print(f"Loaded {len(EVAL_EXAMPLES)} eval examples.")

Loaded 6 eval examples.


## 4.  A small local eval runner

Before we wire scorers, we need something that runs the graph against each example and feeds the result + the expected output into each scorer.

This is what LangSmith's `evaluate()` does internally — we just inline a tiny version so the lab works without LangSmith.

In [6]:
def run_eval(app, examples, scorers, label="run"):
    """Tiny local eval runner. Returns one row per example × scorer."""
    rows = []
    for ex in examples:
        run_state = app.invoke(make_initial_state(ex["inputs"]["ticket"]))
        # `run` is a dict that mimics what LangSmith hands to scorers
        run = {"outputs": run_state}
        for scorer in scorers:
            score = scorer(run, ex)
            rows.append({
                "example": ex["id"],
                "scorer":  score["key"],
                "value":   score["score"],
                **{k: v for k, v in score.items() if k not in ("key", "score")},
            })
    return rows


def report(rows, label="baseline"):
    """Pretty-print a results table + aggregate."""
    print(f"\n=== {label} ===")
    by_scorer = {}
    for r in rows:
        by_scorer.setdefault(r["scorer"], []).append(r)
    # Per-example table
    examples = sorted({r["example"] for r in rows})
    scorers  = sorted(by_scorer)
    header = "  example  " + "  ".join(f"{s:>14}" for s in scorers)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for e in examples:
        cells = []
        for s in scorers:
            match = next((r for r in by_scorer[s] if r["example"] == e), None)
            val = match["value"] if match else "?"
            if isinstance(val, float):
                cells.append(f"{val:>14.2f}")
            else:
                cells.append(f"{str(val):>14}")
        print(f"  {e:9} " + "  ".join(cells))
    # Aggregates
    print()
    for s, rs in by_scorer.items():
        vals = [r["value"] for r in rs if isinstance(r["value"], (int, float))]
        if vals:
            print(f"  {s:>14}  mean = {sum(vals) / len(vals):.2f}   "
                  f"pass-rate = {sum(1 for v in vals if v >= 0.99) / len(vals):.0%}")
    return by_scorer

## 5.  Scorer 1 — Category match

In [7]:
def score_category(run, example):
    """Did Classifier route this ticket to the right bucket?"""
    try:
        run_outputs = run.outputs
    except AttributeError:
        run_outputs = run.get("outputs", {})

    try:
        example_outputs = example.outputs
    except AttributeError:
        example_outputs = example.get("outputs", {})

    pred = run_outputs["category"]
    want = example_outputs["category"]
    return {"key": "category_match", "score": int(pred == want)}

## Scorer 2 — Format / citations

In [8]:
def score_format(run, example):
    """Did Drafter cite the KB articles it should have, with no template leakage?"""
    try:
        run_outputs = run.outputs
    except AttributeError:
        run_outputs = run.get("outputs", {})

    try:
        example_outputs = example.outputs
    except AttributeError:
        example_outputs = example.get("outputs", {})

    draft_text = run_outputs["draft"]

    leaks = ["{ticket}", "{context}", "{policies}"]
    has_leak = any(t in draft_text for t in leaks)

    must_cite = example_outputs.get("must_cite", [])
    missing = [c for c in must_cite if c not in draft_text]

    return {
        "key":   "format_ok",
        "score": int(not has_leak and not missing),
        "leaks": has_leak,
        "missing_citations": missing,
    }

## Scorer 3 — Helpfulness (LLM-as-judge)

In [9]:
JUDGE_PROMPT = """
You are a strict quality reviewer for customer support replies.

Score the REPLY from 1 to 5:
  5 = correct, on-policy, addresses the ticket directly, professional tone
  4 = mostly correct, minor issues with completeness or tone
  3 = correct topic but generic or unhelpful
  2 = on topic but missing key information from the policies
  1 = wrong, off-policy, or fabricated information

Return JSON only:
{{"score": <int 1-5>, "reason": "<one sentence>"}}

TICKET:
{ticket}

POLICIES THAT SHOULD INFORM THE REPLY:
{must_mention}

REPLY:
{draft}
"""

def judge_helpfulness(run, example):
    """LLM-as-judge — fuzzy quality score (helpfulness 1-5, normalized to 0-1)."""
    try:
        example_inputs = example.inputs
    except AttributeError:
        example_inputs = example.get("inputs", {})

    try:
        example_outputs = example.outputs
    except AttributeError:
        example_outputs = example.get("outputs", {})

    try:
        run_outputs = run.outputs
    except AttributeError:
        run_outputs = run.get("outputs", {})

    prompt = JUDGE_PROMPT.format(
        ticket=example_inputs["ticket"],
        must_mention=", ".join(example_outputs.get("must_mention", [])),
        draft=run_outputs["draft"],
    )
    response = llm.invoke(prompt)
    parsed = parse_json(response.content)
    return {
        "key":    "helpfulness",
        "score":  parsed["score"] / 5.0,
        "reason": parsed.get("reason", ""),
    }

## 6.  Run all three scorers against the baseline

This gives us our reference numbers — the bar that any change has to beat (or at least match).

In [10]:
SCORERS = [score_category, score_format, judge_helpfulness]

print("Running baseline...")
baseline_rows = run_eval(app, EVAL_EXAMPLES, SCORERS, label="baseline")
baseline_aggs = report(baseline_rows, label="BASELINE")

Running baseline...

=== BASELINE ===
  example  category_match       format_ok     helpfulness
  -------------------------------------------------------
  E-001                  1               1            1.00
  E-002                  1               1            1.00
  E-003                  1               1            1.00
  E-004                  1               1            1.00
  E-005                  1               1            1.00
  E-006                  0               1            0.40

  category_match  mean = 0.83   pass-rate = 83%
       format_ok  mean = 1.00   pass-rate = 100%
     helpfulness  mean = 0.90   pass-rate = 83%


## 7.  Plant a regression

Someone "improves" the Drafter prompt by making it shorter. They don't run the eval before merging.

Below, we swap in the bad prompt and re-run the suite. Watch what happens to the helpfulness score and the citation checks.

In [11]:
BAD_DRAFT_PROMPT = """\
Reply to this ticket. Be brief.

TICKET: {ticket}
"""


def bad_draft(state):
    """The 'improved' Drafter — no context, no instructions, no citations."""
    prompt = BAD_DRAFT_PROMPT.format(ticket=state["ticket"])
    r = llm.invoke(prompt)
    return {"draft": r.content, "revisions": [r.content]}


# Build a new app with the bad drafter
bad_app = build_graph(draft_fn=bad_draft)

print("Running regression candidate...")
regression_rows = run_eval(bad_app, EVAL_EXAMPLES, SCORERS, label="regression")
regression_aggs = report(regression_rows, label="REGRESSION CANDIDATE")

Running regression candidate...

=== REGRESSION CANDIDATE ===
  example  category_match       format_ok     helpfulness
  -------------------------------------------------------
  E-001                  1               0            0.40
  E-002                  1               0            0.80
  E-003                  1               0            0.80
  E-004                  1               0            0.80
  E-005                  1               0            0.80
  E-006                  0               1            0.80

  category_match  mean = 0.83   pass-rate = 83%
       format_ok  mean = 0.17   pass-rate = 17%
     helpfulness  mean = 0.73   pass-rate = 0%


## 8.  Compare baseline vs regression

The aggregate numbers should tell a clear story: the regression candidate underperforms the baseline on at least one of the scorers.

In a real eval pipeline (LangSmith or otherwise), this comparison would block merges automatically.

In [12]:
def aggregate(rows):
    by_scorer = {}
    for r in rows:
        by_scorer.setdefault(r["scorer"], []).append(r["value"])
    return {s: sum(vs) / len(vs) for s, vs in by_scorer.items()}


base = aggregate(baseline_rows)
reg  = aggregate(regression_rows)

print(f"  {'scorer':>20s}   baseline   candidate   delta")
print("  " + "-" * 56)
for s in sorted(base):
    delta = reg[s] - base[s]
    mark = "✓" if delta >= -0.05 else "✗"
    print(f"  {s:>20s}   {base[s]:8.2f}    {reg[s]:8.2f}   {delta:+.2f}  {mark}")

print()
worst = max((s for s in base), key=lambda s: base[s] - reg[s])
print(f"Worst regression:  {worst}  (baseline {base[worst]:.2f} → candidate {reg[worst]:.2f})")
print("\n→ Block the merge until the drafter prompt change is rolled back or improved.")

                scorer   baseline   candidate   delta
  --------------------------------------------------------
        category_match       0.83        0.83   +0.00  ✓
             format_ok       1.00        0.17   -0.83  ✗
           helpfulness       0.90        0.73   -0.17  ✗

Worst regression:  format_ok  (baseline 1.00 → candidate 0.17)

→ Block the merge until the drafter prompt change is rolled back or improved.


## 9.  Same thing in LangSmith (optional)

If you have a `LANGSMITH_API_KEY` set, you can run the same eval against a hosted dataset using `langsmith.evaluation.evaluate()`. The scorers above plug in unchanged — that's the whole point of keeping them as plain functions.

This cell is optional — skip it if you're staying local.

In [13]:
try:
    if not os.environ.get("LANGSMITH_API_KEY"):
        raise RuntimeError("No LANGSMITH_API_KEY set — skipping hosted run.")

    client = Client()
    dataset_name = "triage-eval-set-v1"

    # Create dataset if it doesn't exist
    try:
        ds = client.read_dataset(dataset_name=dataset_name)
    except Exception:
        ds = client.create_dataset(dataset_name=dataset_name)
        for ex in EVAL_EXAMPLES:
            client.create_example(
                inputs=ex["inputs"],
                outputs=ex["outputs"],
                dataset_id=ds.id,
            )

    results = evaluate(
        lambda inputs: app.invoke(make_initial_state(inputs["ticket"])),
        data=dataset_name,
        evaluators=[score_category, score_format, judge_helpfulness],
        experiment_prefix="baseline",
    )
    print("Hosted experiment created. Open LangSmith to compare runs.")
except Exception as e:
    print(f"Skipping hosted eval: {e}")

View the evaluation results for experiment: 'baseline-104f45d4' at:
https://smith.langchain.com/o/874609a8-0046-4fda-91ab-4f6119540d23/datasets/97a95815-fca0-429c-bdc6-cc88cb991e6f/compare?selectedSessions=c82d9d71-ee3e-45fa-834c-870282ff278f




0it [00:00, ?it/s]

Hosted experiment created. Open LangSmith to compare runs.


## Wrap up

You now have an eval pipeline that:

- ✅ Runs over a curated dataset
- ✅ Applies rule-based + LLM-as-judge scorers
- ✅ Catches a planted regression with quantifiable numbers
- ✅ Works locally and (optionally) in LangSmith

This is the safety net every production agent system needs.

**Up next:** Capstone — pick ONE optimization, apply it, and re-run this exact suite to prove it actually helped.